In [1]:
STUDENT_ID = "TUPM-26-1227"       
SEED_DIGIT = int(STUDENT_ID[-1])

TARGET_YEAR = 2015 + SEED_DIGIT
print("SEED_DIGIT :", SEED_DIGIT)
print("Target year:", TARGET_YEAR)

SEED_DIGIT : 7
Target year: 2022


In [ ]:
import os, csv, json, math, shutil, filecmp

def locate_file(filename, keyword):
    target = filename.lower()
    partial = None
    for root, dirs, files in os.walk("."):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("output", "backup")]
        for f in files:
            if f.lower() == target:
                return os.path.join(root, f)
            if partial is None and keyword in f.lower() and f.lower().endswith(".csv"):
                partial = os.path.join(root, f)
    if partial:
        return partial
    raise FileNotFoundError(
        f"Could not find {filename} under {os.getcwd()}. "
        f"Place it next to this notebook or set DATASET_PATH manually.")

def find_col(header, key, exact=False, required=True):
    for i, name in enumerate(header):
        n = name.strip().lower()
        if (n == key) if exact else (key in n):
            return i
    if required:
        raise ValueError(f"Column containing '{key}' not found in header: {header}")
    return None

def to_float(text):
    try:
        v = float(str(text).strip().replace(",", ""))
    except (ValueError, TypeError):
        return None
    return v if math.isfinite(v) and v >= 0 else None

DATASET_PATH = None            
if DATASET_PATH is None:
    DATASET_PATH = locate_file("Global_Cybersecurity_Threats_2015-2024.csv", "cybersecurity")
print("Dataset:", DATASET_PATH)

for d in ("output", "backup"):
    os.makedirs(d, exist_ok=True)

FILTERED_CSV = os.path.join("output", f"incidents_{TARGET_YEAR}.csv")
SUMMARY_JSON = os.path.join("output", f"incident_summary_{TARGET_YEAR}.json")
BACKUP_JSON  = os.path.join("backup", f"incident_summary_{TARGET_YEAR}_backup.json")
REPORT_TXT   = os.path.join("output", f"incident_report_{TARGET_YEAR}.txt")

Dataset: .\data\Global_Cybersecurity_Threats_2015-2024.csv


In [ ]:
total_records = 0
skipped_rows = 0
selected = []                    

with open(DATASET_PATH, "r", newline="", encoding="utf-8-sig") as src, \
    open(FILTERED_CSV, "w", newline="", encoding="utf-8") as out:

    reader = csv.reader(src)
    writer = csv.writer(out)
    header = next(reader)
    i_year    = find_col(header, "year", exact=True)
    i_attack  = find_col(header, "attack type")
    i_loss    = find_col(header, "financial loss")
    i_country = find_col(header, "country", required=False)
    max_idx = max(i for i in (i_year, i_attack, i_loss) if i is not None)

    writer.writerow(header)

    for row in reader:
        if not any(cell.strip() for cell in row):
            continue
        total_records += 1
        if len(row) <= max_idx:
            skipped_rows += 1
            continue
        try:
            year = int(float(row[i_year].strip()))
        except ValueError:
            skipped_rows += 1
            continue
        if year != TARGET_YEAR:
            continue

        writer.writerow(row)
        selected.append((total_records, to_float(row[i_loss]), row))

incidents = len(selected)
print("Rows read:", total_records, "| malformed/invalid-year skipped:", skipped_rows)
print("Incidents in", TARGET_YEAR, ":", incidents)

Rows read: 3000 | malformed/invalid-year skipped: 0
Incidents in 2022 : 318


In [ ]:
attack_counts = {}
for _, _, row in selected:
    attack = row[i_attack].strip() or "Unknown"
    attack_counts[attack] = attack_counts.get(attack, 0) + 1

valid = [s for s in selected if s[1] is not None]           
invalid_losses = incidents - len(valid)
total_loss = round(sum(s[1] for s in valid), 2)
avg_loss = round(total_loss / len(valid), 2) if valid else None

ranked = sorted(valid, key=lambda s: -s[1])
top5 = ranked[:5]

def record_dict(rownum, loss, row):
    d = {"dataset_row": rownum}
    d.update({header[i].strip(): row[i].strip() for i in range(len(header)) if i < len(row)})
    return d

top5_records = [record_dict(*s) for s in top5]
print("Invalid/empty loss values:", invalid_losses)
print("Total loss:", total_loss, "| Average:", avg_loss)

Invalid/empty loss values: 0
Total loss: 15870.86 | Average: 49.91


In [5]:
summary = {
    "student_id": STUDENT_ID,
    "target_year": TARGET_YEAR,
    "total_records_in_dataset": total_records,
    "incidents_in_target_year": incidents,
    "count_per_attack_type": attack_counts,
    "total_financial_loss_million_usd": total_loss,
    "average_financial_loss_million_usd": avg_loss,
    "invalid_loss_values_excluded": invalid_losses,
    "five_costliest_incidents": top5_records,
    "filtered_csv_file": FILTERED_CSV,
}
with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

backup_ok = False
try:
    shutil.copy2(SUMMARY_JSON, BACKUP_JSON)
    backup_ok = os.path.isfile(BACKUP_JSON) and filecmp.cmp(SUMMARY_JSON, BACKUP_JSON, shallow=False)
except (FileNotFoundError, PermissionError) as e:
    print("Backup failed:", e)
print("Saved:", SUMMARY_JSON)
print("Backup verified (exists and identical):", backup_ok)

Saved: output\incident_summary_2022.json
Backup verified (exists and identical): True


In [6]:
size = lambda p: os.path.getsize(p) if os.path.isfile(p) else 0

lines = [f"CYBERSECURITY INCIDENT REPORT - {TARGET_YEAR}", "=" * 50,
        f"Total records in dataset : {total_records}",
        f"Incidents in {TARGET_YEAR}        : {incidents}", "",
        "Incident count per attack type:"]
lines += [f"  {k}: {v}" for k, v in sorted(attack_counts.items(), key=lambda kv: -kv[1])] or ["  (none)"]
lines += ["", "Financial loss (USD millions):",
        f"  Total   : {total_loss}",
        f"  Average : {avg_loss if avg_loss is not None else 'N/A'}",
        f"  Invalid/empty loss values excluded: {invalid_losses}", "",
        "Five costliest incidents:"]
if top5:
    for rank, (rownum, loss, row) in enumerate(top5, 1):
        country = row[i_country].strip() if i_country is not None else ""
        lines.append(f"  {rank}. Row {rownum} | {row[i_attack].strip()} | {country} | ${loss}M")
else:
    lines.append("  (no incidents with a valid loss value)")
lines += ["", "Output files:",
        f"  Filtered CSV : {FILTERED_CSV} ({size(FILTERED_CSV)} bytes)",
        f"  JSON summary : {SUMMARY_JSON} ({size(SUMMARY_JSON)} bytes)",
        f"  JSON backup  : {BACKUP_JSON} ({size(BACKUP_JSON)} bytes) - verified: {backup_ok}"]

with open(REPORT_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))

CYBERSECURITY INCIDENT REPORT - 2022
Total records in dataset : 3000
Incidents in 2022        : 318

Incident count per attack type:
  Malware: 64
  Phishing: 57
  Ransomware: 56
  DDoS: 54
  SQL Injection: 46
  Man-in-the-Middle: 41

Financial loss (USD millions):
  Total   : 15870.86
  Average : 49.91
  Invalid/empty loss values excluded: 0

Five costliest incidents:
  1. Row 34 | Phishing | Japan | $98.47M
  2. Row 885 | SQL Injection | USA | $98.46M
  3. Row 449 | Phishing | Brazil | $98.06M
  4. Row 851 | Phishing | UK | $97.56M
  5. Row 1692 | Malware | Japan | $97.32M

Output files:
  Filtered CSV : output\incidents_2022.csv (27077 bytes)
  JSON summary : output\incident_summary_2022.json (3213 bytes)
  JSON backup  : backup\incident_summary_2022_backup.json (3213 bytes) - verified: True


In [7]:
counts_line = ", ".join(f"{k}: {v}" for k, v in sorted(attack_counts.items(), key=lambda kv: -kv[1])) or "none"
top5_line = "; ".join(
    f"{row[i_attack].strip()}" + (f"/{row[i_country].strip()}" if i_country is not None else "") + f" ${loss}M (row {rn})"
    for rn, loss, row in top5) or "none"

print(f"Target year                      : {TARGET_YEAR}")
print(f"Total records in dataset         : {total_records}")
print(f"Incidents in target year         : {incidents}")
print(f"Count per attack type            : {counts_line}")
print(f"Total financial loss (USD M)     : {total_loss}")
print(f"Average financial loss (USD M)   : {avg_loss if avg_loss is not None else 'N/A'}")
print(f"Five costliest incidents         : {top5_line}")
print(f"Filtered CSV file                : {FILTERED_CSV}")
print(f"JSON summary file                : {SUMMARY_JSON}")
print(f"Backup verification              : {'Verified - ' + BACKUP_JSON if backup_ok else 'FAILED'}")
print(f"Report file                      : {REPORT_TXT}")

Target year                      : 2022
Total records in dataset         : 3000
Incidents in target year         : 318
Count per attack type            : Malware: 64, Phishing: 57, Ransomware: 56, DDoS: 54, SQL Injection: 46, Man-in-the-Middle: 41
Total financial loss (USD M)     : 15870.86
Average financial loss (USD M)   : 49.91
Five costliest incidents         : Phishing/Japan $98.47M (row 34); SQL Injection/USA $98.46M (row 885); Phishing/Brazil $98.06M (row 449); Phishing/UK $97.56M (row 851); Malware/Japan $97.32M (row 1692)
Filtered CSV file                : output\incidents_2022.csv
JSON summary file                : output\incident_summary_2022.json
Backup verification              : Verified - backup\incident_summary_2022_backup.json
Report file                      : output\incident_report_2022.txt
